# 📑 Daily Loan Report
 
## 🔗 Links
- **[BRD Daily Loan - TBD](xx)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket](https://cepalhellas.atlassian.net/jira/software/c/projects/TXN/list?jql=project%20%3D%20TXN%20ORDER%20BY%20cf%5B10019%5D%20ASC&selectedIssue=TXN-88)**  
  Details about the story ticket, including tasks and progress.
 
- **[ASK / UAT Findings Ticket](https://cepalhellas.atlassian.net/browse/ASK-5202)**  
  A summary of findings from UAT / ASK Ticket.
 
---


In [0]:
import os
import sys

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/DailyLoan/", 1)[0]
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname

env_prefix = get_env_prefix()

## Input Parameters
**report_args:** it is a json which contains all the appropriate parameters to run the report
`
{  
    "portfolio_name": "GALENE",
    "output_file_prefix": "Galene_DailyLoan_",
    "report_output": "/Volumes/qa-cdh-delivery/investorreports/dailyloan/galene/"
}

**run_date:** is the date value to run the report. The run_date is empty when it is called by the orchestrator and that means "today"

In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

import os

env = env_prefix

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
from io import BytesIO
import pandas as pd
import json
from datetime import datetime
from pytz import timezone

In [0]:
athens_tz = timezone('Europe/Athens')
try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("'", '"')
except Exception as e:
    raise ValueError("Missing required input: report_args")
try:    
    report_name = dbutils.widgets.get("report_name")
except Exception as e:
    raise ValueError("Missing required input: report_name")
print(f"Report {report_name} is started")

run_date_raw = dbutils.widgets.get("run_date").strip()
if run_date_raw:
    run_date = run_date_raw
else:
    run_date = str(spark.sql(f"SELECT MAX(BusinessDate) FROM `{env}-cdh`.derivative.dexposure").collect()[0][0])

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

# Read JSON
json_data = json.loads(report_args)
# Access the values and store them into variables
if "portfolio_name" not in json_data: raise KeyError("'portfolio_name' key is missing in the report_args JSON data.")
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")
portfolioName = "".join(json_data["portfolio_name"])
output_file_prefix = "".join(json_data["output_file_prefix"])
report_output = "".join(json_data["report_output"])
# Resolve full export path from SSM if relative (same pattern as MegaDataTape)
if not report_output.startswith("/Volumes"):
    from aws_utils import get_scope_key_names, get_aws_credentials, connect_to_aws_and_access_ssm
    import boto3
    _secrets_scope = "aws-databricks"
    _aws_keys = get_scope_key_names(_secrets_scope)
    _aws_key = _aws_keys[0] if _aws_keys else None
    _aws_secret_key = get_aws_credentials(_secrets_scope, _aws_key)
    _ssm_client = boto3.client("ssm", region_name="eu-central-1",
                               aws_access_key_id=_aws_key,
                               aws_secret_access_key=_aws_secret_key)
    export_base_path = connect_to_aws_and_access_ssm(
        "/reports/exports/s3_main_path", _ssm_client, _aws_secret_key
    ).replace("envprefix", env)
    report_output = export_base_path + report_output
print(f"Report {report_name} is running with the following arguments: report_args = {report_args}, report_name = {report_name}, portfolio_name = {portfolioName}, output_file_prefix = {output_file_prefix}, run_date = {run_date}")
print(f"now is: {now_datetime_str}")

## Import Tables

In [0]:
ExchangeRate = spark.sql(
f'''
SELECT 
        E.BusinessDate  
      , E.OriginalCurrency                   
      , E.ExchangeRate                                       
FROM `{env}-cdh`.reference.exchangerates as E
where E.businessdate = '{run_date}'
'''
)
ExchangeRate.createOrReplaceTempView("ExchangeRate")

Final = spark.sql(f"""                          
select 
        '{run_date}' as ReportingDate
    ,   de.ExposureKey As AccountNo
    ,   de.CDI AS BorrowerID
    ,   CAST(de.OutstandingTotalBalance AS DOUBLE) AS TotalExposure  
    ,   CAST(de.OutstandingTotalBalance / coalesce( ER.ExchangeRate, 1) AS DOUBLE) AS TotalExposureEur
    ,   CAST(de.GBV AS DOUBLE) AS GBV
    ,   CAST(de.GBV / coalesce( ER.ExchangeRate, 1) AS DOUBLE) AS GBVEur
    ,   de.DaysPastDueWithMateriality AS DaysPastDue
    ,   de.ActiveModificationProgramStatus
    ,   de.LastModificationActionType
    ,   de.LastModificationDate
    ,   de.DenouncedDate as DenouncementDate
    ,   de.DenouncementIndicator
    ,   de.L3869Flag
    ,   de.L3869Status
    ,   de.L3869SubmissionDate
    ,   de.L3869ProvisionalCourtDecisionDate
    ,   de.PrincipalAtDefault
    ,   CAST(de.PrincipalAtDefault / coalesce( ER.ExchangeRate, 1) AS DOUBLE) AS  PrincipalAtDefaultEur
    ,   de.InterestAtDefault
    ,   de.AmountDueAtDefault
    ,   CAST(de.DueAmount AS DOUBLE) as BalanceInArrears      
    ,   CASE WHEN  de.ProductCode in ('40584', '43619')  then true else false end as MLTCondWoffFlag
 FROM `{env}-cdh`.derivative.dexposure as de
 LEFT JOIN ExchangeRate  AS ER ON de.Currency = ER.OriginalCurrency
 WHERE de.BusinessDate = '{run_date}'
 AND de.PortfolioName = '{portfolioName}'
 AND de.IsActiveFlag = true
""")
Final.createOrReplaceTempView("Final")

## Final Export

In [0]:
from export_utils import export_data
from helpers import archive_existing_files
from pyspark.sql import functions as F

final_pdf = Final.toPandas()

output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer) as writer:
    final_pdf.to_excel(writer, sheet_name=f"DailyLoan_{portfolioName}", index=False)

output_path = f"{report_output}/{output_file_prefix}{run_date}.xlsx"

# Archive existing files before writing new export
os.makedirs(os.path.dirname(output_path), exist_ok=True)
archive_existing_files(os.path.dirname(output_path))

with open(output_path, "wb") as f:
    f.write(output_buffer.getvalue())